# Day 12 — Gold Dimensional: Aggregated Marts

Builds all 9 buildable aggregated (presentation) marts from the dimensional fact + dim tables.
One mart (MartFleetUtilisation) is skipped — source data not available.

| Mart | Grain | Source Facts/Dims | Use Case |
|---|---|---|---|
| MartRevenueByStation | station × month | FactChargingSession + DimStation | Revenue reporting |
| MartRevenueByTariff | tariff × month | FactChargingSession + DimTariff | Tariff performance |
| MartCustomerLifetime | customer | FactChargingSession + DimCustomer | CLV, churn |
| MartChargerPerformance | charger × month | FactChargingSession + FactEnergyConsumption + FactDeviceTelemetry | Ops |
| MartMaintenanceKPI | station × month | FactMaintenance + DimStation | SLA/ops |
| MartComplaintAnalysis | station × month | FactComplaints + DimStation | CX |
| MartEnergyEfficiency | charger × month | FactEnergyConsumption + FactDeviceTelemetry | Grid efficiency |
| MartPaymentReconciliation | month | FactChargingSession + FactPayments | Finance reconciliation |
| MartStationHealthDashboard | station × month | FactStationUtilisation + FactMaintenance + FactComplaints | Ops dashboard |

In [ ]:
# Cell 1 — Imports
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from datetime import datetime, timezone

print('Imports OK')

In [ ]:
# Cell 2 — Constants
GOLD_DIM   = '/Volumes/dbw_ev_intelligence_dev/default/gold-volume/dimensional/dims'
GOLD_FACT  = '/Volumes/dbw_ev_intelligence_dev/default/gold-volume/dimensional/facts'
GOLD_MART  = '/Volumes/dbw_ev_intelligence_dev/default/gold-volume/dimensional/marts'
PIPELINE   = 'pl_gold_dimensional_marts_v1'
RUN_TS     = datetime.now(timezone.utc).strftime('%Y-%m-%dT%H:%M:%SZ')

def read_dim(name):
    return spark.read.format('delta').load(f'{GOLD_DIM}/{name}')

def read_fact(name):
    return spark.read.format('delta').load(f'{GOLD_FACT}/{name}')

def write_mart(df, mart_name, partition_cols=None):
    path = f'{GOLD_MART}/{mart_name}'
    writer = df.write.format('delta').mode('overwrite').option('overwriteSchema', 'true')
    if partition_cols:
        writer = writer.partitionBy(*partition_cols)
    writer.save(path)
    print(f'  {mart_name:<35} {df.count():>8} rows  -> {path}')

print(f'Gold marts : {GOLD_MART}')
print(f'RUN_TS     : {RUN_TS}')

In [ ]:
# Cell 3 — Load fact and dim tables
fact_session     = read_fact('FactChargingSession')
fact_energy      = read_fact('FactEnergyConsumption')
fact_payments    = read_fact('FactPayments')
fact_maintenance = read_fact('FactMaintenance')
fact_telemetry   = read_fact('FactDeviceTelemetry')
fact_complaints  = read_fact('FactComplaints')
fact_utilisation = read_fact('FactStationUtilisation')

dim_station  = read_dim('DimStation')
dim_charger  = read_dim('DimCharger').filter(F.col('is_current') == True)
dim_customer = read_dim('DimCustomer').filter(F.col('is_current') == True)
dim_tariff   = read_dim('DimTariff').filter(F.col('is_current') == True)

print('Facts and dims loaded')

In [ ]:
# Cell 4 — MartRevenueByStation
# Grain: 1 row per station per month.
# Metrics: total sessions, energy delivered, revenue, avg session duration, avg cost per session.

sta_attrs = dim_station.select(
    'station_key', 'station_name', 'state_code', 'country_name', 'total_chargers'
)

mart_revenue_station = (
    fact_session
    .groupBy('station_key', 'session_year', 'session_month')
    .agg(
        F.count('session_id').alias('total_sessions'),
        F.sum('energy_kwh').cast('decimal(14,4)').alias('total_energy_kwh'),
        F.sum('total_payment_aud').cast('decimal(14,2)').alias('total_revenue_aud'),
        F.sum('expected_cost_aud').cast('decimal(14,2)').alias('total_expected_cost_aud'),
        F.avg('duration_min').cast('decimal(8,2)').alias('avg_duration_min'),
        F.avg('total_payment_aud').cast('decimal(10,2)').alias('avg_revenue_per_session_aud'),
        F.avg('energy_kwh').cast('decimal(10,4)').alias('avg_energy_per_session_kwh'),
        F.countDistinct('customer_key').alias('unique_customers'),
        F.sum(F.when(F.col('reconciliation_status') == 'MISMATCH', 1).otherwise(0)).alias('mismatch_count'),
    )
    .join(sta_attrs, on='station_key', how='left')
    .withColumn('revenue_per_charger_aud',
        F.when(
            F.col('total_chargers').isNotNull() & (F.col('total_chargers') > 0),
            (F.col('total_revenue_aud') / F.col('total_chargers')).cast('decimal(12,2)')
        )
    )
    .withColumn('gold_ingested_at', F.lit(RUN_TS).cast('timestamp'))
    .withColumn('gold_pipeline',    F.lit(PIPELINE))
    .select(
        'station_key', 'station_name', 'state_code', 'country_name',
        'session_year', 'session_month',
        'total_sessions', 'unique_customers',
        'total_energy_kwh', 'avg_energy_per_session_kwh',
        'total_revenue_aud', 'total_expected_cost_aud',
        'avg_revenue_per_session_aud', 'avg_duration_min',
        'total_chargers', 'revenue_per_charger_aud',
        'mismatch_count',
        'gold_ingested_at', 'gold_pipeline',
    )
)

write_mart(mart_revenue_station, 'MartRevenueByStation', partition_cols=['session_year', 'session_month'])

In [ ]:
# Cell 5 — MartRevenueByTariff
# Grain: 1 row per tariff per month.
# Shows how each pricing plan performs: sessions, revenue, avg energy.

tar_attrs = dim_tariff.select(
    'tariff_key', 'tariff_name', 'rate_per_kwh', 'rate_per_min', 'currency'
)

mart_revenue_tariff = (
    fact_session
    .groupBy('tariff_key', 'session_year', 'session_month')
    .agg(
        F.count('session_id').alias('total_sessions'),
        F.sum('energy_kwh').cast('decimal(14,4)').alias('total_energy_kwh'),
        F.sum('total_payment_aud').cast('decimal(14,2)').alias('total_revenue_aud'),
        F.avg('duration_min').cast('decimal(8,2)').alias('avg_duration_min'),
        F.avg('cost_per_kwh').cast('decimal(8,4)').alias('avg_cost_per_kwh'),
        F.countDistinct('customer_key').alias('unique_customers'),
        F.countDistinct('station_key').alias('stations_used'),
        F.sum(F.when(F.col('reconciliation_status') == 'MATCH', 1).otherwise(0)).alias('match_count'),
        F.sum(F.when(F.col('reconciliation_status') == 'MISMATCH', 1).otherwise(0)).alias('mismatch_count'),
    )
    .join(tar_attrs, on='tariff_key', how='left')
    .withColumn('match_rate_pct',
        F.when(F.col('total_sessions') > 0,
            (F.col('match_count') / F.col('total_sessions') * 100).cast('decimal(6,2)')
        )
    )
    .withColumn('gold_ingested_at', F.lit(RUN_TS).cast('timestamp'))
    .withColumn('gold_pipeline',    F.lit(PIPELINE))
    .select(
        'tariff_key', 'tariff_name', 'rate_per_kwh', 'rate_per_min', 'currency',
        'session_year', 'session_month',
        'total_sessions', 'unique_customers', 'stations_used',
        'total_energy_kwh', 'total_revenue_aud',
        'avg_duration_min', 'avg_cost_per_kwh',
        'match_count', 'mismatch_count', 'match_rate_pct',
        'gold_ingested_at', 'gold_pipeline',
    )
)

write_mart(mart_revenue_tariff, 'MartRevenueByTariff', partition_cols=['session_year', 'session_month'])

In [ ]:
# Cell 6 — MartCustomerLifetime
# Grain: 1 row per customer (lifetime aggregate — no time partition).
# Metrics: total sessions, total spend, first/last session date, avg session metrics.

cust_attrs = dim_customer.select(
    'customer_key', 'full_name', 'email', 'city', 'state', 'country', 'signup_date'
)

mart_customer_ltv = (
    fact_session
    .groupBy('customer_key')
    .agg(
        F.count('session_id').alias('total_sessions'),
        F.sum('total_payment_aud').cast('decimal(14,2)').alias('total_spend_aud'),
        F.sum('energy_kwh').cast('decimal(14,4)').alias('total_energy_kwh'),
        F.avg('duration_min').cast('decimal(8,2)').alias('avg_session_min'),
        F.avg('energy_kwh').cast('decimal(10,4)').alias('avg_session_kwh'),
        F.avg('total_payment_aud').cast('decimal(10,2)').alias('avg_spend_per_session_aud'),
        F.min('session_date').alias('first_session_date'),
        F.max('session_date').alias('last_session_date'),
        F.countDistinct('station_key').alias('stations_visited'),
        F.countDistinct('charger_key').alias('chargers_used'),
        F.sum(F.when(F.col('reconciliation_status') == 'MISMATCH', 1).otherwise(0)).alias('payment_mismatches'),
    )
    .join(cust_attrs, on='customer_key', how='left')
    .withColumn('days_since_first_session',
        F.datediff(F.current_date(), F.col('first_session_date'))
    )
    .withColumn('days_since_last_session',
        F.datediff(F.current_date(), F.col('last_session_date'))
    )
    .withColumn('spend_per_day_aud',
        F.when(
            F.col('days_since_first_session').isNotNull() & (F.col('days_since_first_session') > 0),
            (F.col('total_spend_aud') / F.col('days_since_first_session')).cast('decimal(10,4)')
        )
    )
    .withColumn('gold_ingested_at', F.lit(RUN_TS).cast('timestamp'))
    .withColumn('gold_pipeline',    F.lit(PIPELINE))
    .select(
        'customer_key', 'full_name', 'email', 'city', 'state', 'country', 'signup_date',
        'total_sessions', 'total_spend_aud', 'total_energy_kwh',
        'avg_session_min', 'avg_session_kwh', 'avg_spend_per_session_aud',
        'first_session_date', 'last_session_date',
        'days_since_first_session', 'days_since_last_session',
        'spend_per_day_aud',
        'stations_visited', 'chargers_used', 'payment_mismatches',
        'gold_ingested_at', 'gold_pipeline',
    )
)

write_mart(mart_customer_ltv, 'MartCustomerLifetime')
print('\nTop 5 customers by lifetime spend:')
mart_customer_ltv.select('full_name', 'total_sessions', 'total_spend_aud', 'avg_spend_per_session_aud') \
    .orderBy(F.col('total_spend_aud').desc()).show(5, truncate=False)

In [ ]:
# Cell 7 — MartChargerPerformance
# Grain: 1 row per charger per month.
# Combines session revenue (FactChargingSession), energy delivery (FactEnergyConsumption),
# and telemetry (FactDeviceTelemetry) for full charger ops picture.

chg_attrs = dim_charger.select(
    'charger_key', 'charger_id', 'charger_type', 'connector_type', 'max_kw', 'station_id'
)
sta_state = dim_station.select('station_key', 'station_name', 'state_code')

# Monthly session metrics per charger
session_by_charger = (
    fact_session
    .groupBy('charger_key', 'session_year', 'session_month')
    .agg(
        F.count('session_id').alias('total_sessions'),
        F.sum('total_payment_aud').cast('decimal(14,2)').alias('total_revenue_aud'),
        F.sum('energy_kwh').cast('decimal(14,4)').alias('total_energy_kwh'),
        F.avg('duration_min').cast('decimal(8,2)').alias('avg_session_min'),
        F.countDistinct('customer_key').alias('unique_customers'),
    )
)

# Monthly energy consumption metrics per charger
energy_by_charger = (
    fact_energy
    .groupBy('charger_key', 'consumption_year', 'consumption_month')
    .agg(
        F.sum('total_active_min').cast('decimal(12,2)').alias('total_active_min'),
        F.avg('avg_energy_kwh').cast('decimal(10,4)').alias('avg_hourly_kwh'),
    )
)

# Monthly telemetry averages per charger
tele_by_charger = (
    fact_telemetry
    .groupBy('charger_key', 'telemetry_year', 'telemetry_month')
    .agg(
        F.avg('power_efficiency_pct').cast('decimal(6,2)').alias('avg_efficiency_pct'),
        F.avg('device_temp_c').cast('decimal(6,2)').alias('avg_device_temp_c'),
        F.avg('signal_strength_dbm').cast('decimal(6,2)').alias('avg_signal_dbm'),
        F.countDistinct('firmware_ver').alias('firmware_versions_seen'),
    )
)

mart_charger_perf = (
    session_by_charger
    .join(
        energy_by_charger.withColumnRenamed('consumption_year', 'session_year')
                         .withColumnRenamed('consumption_month', 'session_month'),
        on=['charger_key', 'session_year', 'session_month'], how='left'
    )
    .join(
        tele_by_charger.withColumnRenamed('telemetry_year', 'session_year')
                       .withColumnRenamed('telemetry_month', 'session_month'),
        on=['charger_key', 'session_year', 'session_month'], how='left'
    )
    .join(chg_attrs, on='charger_key', how='left')
    .join(sta_state, on=chg_attrs['station_id'] == sta_state['station_key'], how='left')
    .withColumn('gold_ingested_at', F.lit(RUN_TS).cast('timestamp'))
    .withColumn('gold_pipeline',    F.lit(PIPELINE))
    .select(
        'charger_key', 'charger_id', 'charger_type', 'connector_type', 'max_kw',
        'session_year', 'session_month',
        'total_sessions', 'unique_customers',
        'total_energy_kwh', 'total_revenue_aud',
        'avg_session_min', 'avg_hourly_kwh',
        'total_active_min',
        'avg_efficiency_pct', 'avg_device_temp_c',
        'avg_signal_dbm', 'firmware_versions_seen',
        'gold_ingested_at', 'gold_pipeline',
    )
)

write_mart(mart_charger_perf, 'MartChargerPerformance', partition_cols=['session_year', 'session_month'])

In [ ]:
# Cell 8 — MartMaintenanceKPI
# Grain: 1 row per station per month.
# Metrics: total events, avg resolution hours, events by severity, unresolved count.

sta_attrs2 = dim_station.select('station_key', 'station_name', 'state_code')

mart_maint_kpi = (
    fact_maintenance
    .groupBy('station_key', 'event_year', 'event_month')
    .agg(
        F.count('event_id').alias('total_events'),
        F.sum(F.when(F.col('is_resolved') == True, 1).otherwise(0)).alias('resolved_events'),
        F.sum(F.when(F.col('is_resolved') == False, 1).otherwise(0)).alias('unresolved_events'),
        F.avg(F.when(F.col('is_resolved') == True, F.col('resolution_hours'))).cast('decimal(8,2)').alias('avg_resolution_hours'),
        F.max('resolution_hours').cast('decimal(8,2)').alias('max_resolution_hours'),
        F.sum(F.when(F.col('severity') == 'critical', 1).otherwise(0)).alias('critical_events'),
        F.sum(F.when(F.col('severity') == 'high', 1).otherwise(0)).alias('high_events'),
        F.sum(F.when(F.col('severity') == 'medium', 1).otherwise(0)).alias('medium_events'),
        F.sum(F.when(F.col('severity') == 'low', 1).otherwise(0)).alias('low_events'),
        F.countDistinct('charger_key').alias('chargers_with_events'),
    )
    .join(sta_attrs2, on='station_key', how='left')
    .withColumn('resolution_rate_pct',
        F.when(F.col('total_events') > 0,
            (F.col('resolved_events') / F.col('total_events') * 100).cast('decimal(6,2)')
        )
    )
    .withColumn('gold_ingested_at', F.lit(RUN_TS).cast('timestamp'))
    .withColumn('gold_pipeline',    F.lit(PIPELINE))
    .select(
        'station_key', 'station_name', 'state_code',
        'event_year', 'event_month',
        'total_events', 'resolved_events', 'unresolved_events',
        'resolution_rate_pct', 'avg_resolution_hours', 'max_resolution_hours',
        'critical_events', 'high_events', 'medium_events', 'low_events',
        'chargers_with_events',
        'gold_ingested_at', 'gold_pipeline',
    )
)

write_mart(mart_maint_kpi, 'MartMaintenanceKPI', partition_cols=['event_year', 'event_month'])

In [ ]:
# Cell 9 — MartComplaintAnalysis
# Grain: 1 row per station per month.
# Shows complaint volume, resolution rate, avg resolution days, by category.

sta_attrs3 = dim_station.select('station_key', 'station_name', 'state_code')

mart_complaints = (
    fact_complaints
    .groupBy('station_key', 'station_id', 'complaint_year', 'complaint_month')
    .agg(
        F.count('complaint_id').alias('total_complaints'),
        F.sum(F.when(F.col('is_resolved') == True, 1).otherwise(0)).alias('resolved_complaints'),
        F.avg(F.when(F.col('is_resolved') == True, F.col('resolution_days'))).cast('decimal(8,2)').alias('avg_resolution_days'),
        F.max('resolution_days').cast('decimal(8,2)').alias('max_resolution_days'),
        F.sum(F.when(F.col('priority') == 'high', 1).otherwise(0)).alias('high_priority_complaints'),
        F.sum(F.when(F.col('priority') == 'critical', 1).otherwise(0)).alias('critical_complaints'),
        F.countDistinct('customer_key').alias('unique_complainants'),
    )
    .join(sta_attrs3, on='station_key', how='left')
    .withColumn('resolution_rate_pct',
        F.when(F.col('total_complaints') > 0,
            (F.col('resolved_complaints') / F.col('total_complaints') * 100).cast('decimal(6,2)')
        )
    )
    .withColumn('gold_ingested_at', F.lit(RUN_TS).cast('timestamp'))
    .withColumn('gold_pipeline',    F.lit(PIPELINE))
    .select(
        'station_key', 'station_name', 'state_code', 'station_id',
        'complaint_year', 'complaint_month',
        'total_complaints', 'resolved_complaints',
        'resolution_rate_pct', 'avg_resolution_days', 'max_resolution_days',
        'high_priority_complaints', 'critical_complaints',
        'unique_complainants',
        'gold_ingested_at', 'gold_pipeline',
    )
)

write_mart(mart_complaints, 'MartComplaintAnalysis', partition_cols=['complaint_year', 'complaint_month'])

In [ ]:
# Cell 10 — MartEnergyEfficiency
# Grain: 1 row per charger per month.
# Combines energy consumption + telemetry to surface grid efficiency and device health trends.

chg_attrs2 = dim_charger.select('charger_key', 'charger_type', 'max_kw', 'station_id')

energy_monthly = (
    fact_energy
    .groupBy('charger_key', 'station_key', 'consumption_year', 'consumption_month')
    .agg(
        F.sum('total_energy_kwh').cast('decimal(14,4)').alias('total_energy_kwh'),
        F.sum('session_count').alias('total_sessions'),
        F.sum('total_duration_min').cast('decimal(12,2)').alias('total_active_min'),
    )
)

tele_monthly = (
    fact_telemetry
    .groupBy('charger_key', 'telemetry_year', 'telemetry_month')
    .agg(
        F.avg('power_efficiency_pct').cast('decimal(6,2)').alias('avg_efficiency_pct'),
        F.max('power_efficiency_pct').cast('decimal(6,2)').alias('peak_efficiency_pct'),
        F.avg('device_temp_c').cast('decimal(6,2)').alias('avg_device_temp_c'),
        F.max('device_temp_c').cast('decimal(6,2)').alias('max_device_temp_c'),
        F.avg('signal_strength_dbm').cast('decimal(6,2)').alias('avg_signal_dbm'),
    )
)

mart_energy_eff = (
    energy_monthly
    .join(
        tele_monthly.withColumnRenamed('telemetry_year', 'consumption_year')
                    .withColumnRenamed('telemetry_month', 'consumption_month'),
        on=['charger_key', 'consumption_year', 'consumption_month'], how='left'
    )
    .join(chg_attrs2, on='charger_key', how='left')
    # Max possible kWh in the month = max_kw × hours_in_month × sessions_that_used_full_power
    # Simpler: theoretical max = max_kw × total_active_hours
    .withColumn('theoretical_max_kwh',
        F.when(
            chg_attrs2['max_kw'].isNotNull() & (chg_attrs2['max_kw'] > 0)
            & F.col('total_active_min').isNotNull(),
            (chg_attrs2['max_kw'] * F.col('total_active_min') / 60).cast('decimal(14,4)')
        )
    )
    .withColumn('delivered_vs_max_pct',
        F.when(
            F.col('theoretical_max_kwh').isNotNull() & (F.col('theoretical_max_kwh') > 0),
            (F.col('total_energy_kwh') / F.col('theoretical_max_kwh') * 100).cast('decimal(6,2)')
        )
    )
    .withColumn('gold_ingested_at', F.lit(RUN_TS).cast('timestamp'))
    .withColumn('gold_pipeline',    F.lit(PIPELINE))
    .select(
        'charger_key', 'charger_type', 'max_kw',
        F.col('consumption_year').alias('report_year'),
        F.col('consumption_month').alias('report_month'),
        'total_sessions', 'total_energy_kwh', 'total_active_min',
        'theoretical_max_kwh', 'delivered_vs_max_pct',
        'avg_efficiency_pct', 'peak_efficiency_pct',
        'avg_device_temp_c', 'max_device_temp_c',
        'avg_signal_dbm',
        'gold_ingested_at', 'gold_pipeline',
    )
)

write_mart(mart_energy_eff, 'MartEnergyEfficiency', partition_cols=['report_year', 'report_month'])

In [ ]:
# Cell 11 — MartPaymentReconciliation
# Grain: 1 row per month (finance-level summary).
# Joins FactChargingSession reconciliation status vs FactPayments actual amounts.

# Monthly session reconciliation
session_recon = (
    fact_session
    .groupBy('session_year', 'session_month')
    .agg(
        F.count('session_id').alias('total_sessions'),
        F.sum('expected_cost_aud').cast('decimal(16,2)').alias('total_expected_aud'),
        F.sum('total_payment_aud').cast('decimal(16,2)').alias('total_billed_aud'),
        F.sum(F.when(F.col('reconciliation_status') == 'MATCH', 1).otherwise(0)).alias('matched_sessions'),
        F.sum(F.when(F.col('reconciliation_status') == 'MISMATCH', 1).otherwise(0)).alias('mismatch_sessions'),
        F.sum(F.when(F.col('reconciliation_status') == 'UNMATCHED', 1).otherwise(0)).alias('unmatched_sessions'),
        F.sum(
            F.when(F.col('reconciliation_status') == 'MISMATCH',
                F.abs(F.col('total_payment_aud') - F.col('expected_cost_aud'))
            ).otherwise(F.lit(0))
        ).cast('decimal(14,2)').alias('total_variance_aud'),
    )
)

# Monthly payments summary
payment_monthly = (
    fact_payments
    .groupBy('payment_year', 'payment_month')
    .agg(
        F.count('payment_id').alias('total_payments'),
        F.sum('amount_aud').cast('decimal(16,2)').alias('total_collected_aud'),
        F.sum('gst_aud').cast('decimal(14,2)').alias('total_gst_aud'),
        F.sum('amount_ex_gst').cast('decimal(16,2)').alias('total_ex_gst_aud'),
        F.sum(F.when(F.col('refund_flag') == True, F.col('amount_aud')).otherwise(F.lit(0))).cast('decimal(14,2)').alias('total_refunded_aud'),
        F.sum(F.when(F.col('refund_flag') == True, 1).otherwise(0)).alias('refund_count'),
    )
)

mart_recon = (
    session_recon
    .join(
        payment_monthly.withColumnRenamed('payment_year', 'session_year')
                       .withColumnRenamed('payment_month', 'session_month'),
        on=['session_year', 'session_month'], how='left'
    )
    .withColumn('match_rate_pct',
        F.when(F.col('total_sessions') > 0,
            (F.col('matched_sessions') / F.col('total_sessions') * 100).cast('decimal(6,2)')
        )
    )
    .withColumn('collection_rate_pct',
        F.when(
            F.col('total_expected_aud').isNotNull() & (F.col('total_expected_aud') > 0),
            (F.col('total_collected_aud') / F.col('total_expected_aud') * 100).cast('decimal(6,2)')
        )
    )
    .withColumn('gold_ingested_at', F.lit(RUN_TS).cast('timestamp'))
    .withColumn('gold_pipeline',    F.lit(PIPELINE))
    .select(
        'session_year', 'session_month',
        'total_sessions', 'matched_sessions', 'mismatch_sessions', 'unmatched_sessions',
        'match_rate_pct',
        'total_expected_aud', 'total_billed_aud', 'total_variance_aud',
        'total_payments', 'total_collected_aud',
        'total_gst_aud', 'total_ex_gst_aud',
        'collection_rate_pct',
        'refund_count', 'total_refunded_aud',
        'gold_ingested_at', 'gold_pipeline',
    )
)

write_mart(mart_recon, 'MartPaymentReconciliation', partition_cols=['session_year', 'session_month'])
print('\nReconciliation monthly summary:')
mart_recon.select(
    'session_year', 'session_month', 'total_sessions',
    'match_rate_pct', 'total_expected_aud', 'total_collected_aud',
    'collection_rate_pct', 'total_variance_aud'
).orderBy('session_year', 'session_month').show(truncate=False)

In [ ]:
# Cell 12 — MartStationHealthDashboard
# Grain: 1 row per station per month.
# Combines utilisation, maintenance load, and complaint volume into a single ops dashboard.

sta_attrs4 = dim_station.select('station_key', 'station_name', 'state_code', 'total_chargers')

# Monthly utilisation per station
util_monthly = (
    fact_utilisation
    .groupBy('station_key', 'utilisation_year', 'utilisation_month')
    .agg(
        F.avg('utilisation_pct').cast('decimal(6,2)').alias('avg_utilisation_pct'),
        F.max('utilisation_pct').cast('decimal(6,2)').alias('peak_utilisation_pct'),
        F.sum('session_count').alias('total_sessions'),
        F.sum('total_energy_kwh').cast('decimal(14,4)').alias('total_energy_kwh'),
    )
)

# Monthly maintenance count per station
maint_monthly = (
    fact_maintenance
    .groupBy('station_key', 'event_year', 'event_month')
    .agg(
        F.count('event_id').alias('maintenance_events'),
        F.sum(F.when(F.col('severity').isin('critical', 'high'), 1).otherwise(0)).alias('critical_high_events'),
        F.avg('resolution_hours').cast('decimal(8,2)').alias('avg_maintenance_resolution_h'),
    )
)

# Monthly complaint count per station
complaint_monthly = (
    fact_complaints
    .groupBy('station_key', 'complaint_year', 'complaint_month')
    .agg(
        F.count('complaint_id').alias('complaint_count'),
        F.sum(F.when(F.col('priority').isin('critical', 'high'), 1).otherwise(0)).alias('critical_high_complaints'),
    )
)

mart_station_health = (
    util_monthly
    .join(
        maint_monthly.withColumnRenamed('event_year', 'utilisation_year')
                     .withColumnRenamed('event_month', 'utilisation_month'),
        on=['station_key', 'utilisation_year', 'utilisation_month'], how='left'
    )
    .join(
        complaint_monthly.withColumnRenamed('complaint_year', 'utilisation_year')
                         .withColumnRenamed('complaint_month', 'utilisation_month'),
        on=['station_key', 'utilisation_year', 'utilisation_month'], how='left'
    )
    .join(sta_attrs4, on='station_key', how='left')
    .withColumn('maintenance_per_charger',
        F.when(
            F.col('total_chargers').isNotNull() & (F.col('total_chargers') > 0),
            (F.coalesce(F.col('maintenance_events'), F.lit(0)) / F.col('total_chargers')).cast('decimal(8,2)')
        )
    )
    .withColumn('gold_ingested_at', F.lit(RUN_TS).cast('timestamp'))
    .withColumn('gold_pipeline',    F.lit(PIPELINE))
    .select(
        'station_key', 'station_name', 'state_code', 'total_chargers',
        F.col('utilisation_year').alias('report_year'),
        F.col('utilisation_month').alias('report_month'),
        'total_sessions', 'total_energy_kwh',
        'avg_utilisation_pct', 'peak_utilisation_pct',
        F.coalesce(F.col('maintenance_events'), F.lit(0)).alias('maintenance_events'),
        F.coalesce(F.col('critical_high_events'), F.lit(0)).alias('critical_high_maintenance'),
        'avg_maintenance_resolution_h', 'maintenance_per_charger',
        F.coalesce(F.col('complaint_count'), F.lit(0)).alias('complaint_count'),
        F.coalesce(F.col('critical_high_complaints'), F.lit(0)).alias('critical_high_complaints'),
        'gold_ingested_at', 'gold_pipeline',
    )
)

write_mart(mart_station_health, 'MartStationHealthDashboard', partition_cols=['report_year', 'report_month'])

In [ ]:
# Cell 13 — Run Summary
marts = [
    'MartRevenueByStation',
    'MartRevenueByTariff',
    'MartCustomerLifetime',
    'MartChargerPerformance',
    'MartMaintenanceKPI',
    'MartComplaintAnalysis',
    'MartEnergyEfficiency',
    'MartPaymentReconciliation',
    'MartStationHealthDashboard',
]
print('=' * 65)
print('GOLD DIMENSIONAL — AGGREGATED MARTS SUMMARY')
print('=' * 65)
for m in marts:
    path = f'{GOLD_MART}/{m}'
    try:
        cnt = spark.read.format('delta').load(path).count()
        print(f'  {m:<35} {cnt:>8} rows')
    except Exception as e:
        print(f'  {m:<35} ERROR: {e}')
print(f'\nRUN_TS : {RUN_TS}')
print('\nSkipped: MartFleetUtilisation — no telematics/fleet source data')